# RAINBOW DQL

In [ ]:
from collections import deque
from IPython import display
import gym
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Dropout, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
    
from src.utils.plots import plot_training_results

%matplotlib inline

In [ ]:
class RainbowDQNAgent():

    def __init__(
        self, env, gamma=0.99, epsilon=1, alpha=0.01, 
        batch_size=64, buffer_size=50000, hidden_neurons=32
    ):
        
        self.env = env
        self.na = env.action_space.n
        self.n_features = env.observation_space.shape[0]
        self.epsilon = epsilon 
        self.gamma = gamma 

        self.batch_size = batch_size
        self.buffer_size = buffer_size
        self.buffer = deque(maxlen=self.buffer_size)
        self.hidden_neurons = hidden_neurons
        
        self.primary_model = self.network()
        self.primary_model.compile(loss="mse", optimizer=Adam(learning_rate=alpha))

        self.target_model = self.network()
        self.update_target()
    
    def network(self):
        inp = Input(shape=(self.n_features, ))
        capa = Dense(int(self.hidden_neurons), activation="relu")(inp)
        capa = Dense(int(self.hidden_neurons), activation="relu")(capa)

        # Standard out
        # out = Dense(self.na, activation="linear")(capa)

        # Dueling:
        adv = Dense(self.na)(capa)
        adv_mean = tf.reduce_mean(adv, 1, keepdims=True)
        adv = tf.subtract(adv, adv_mean)
        v = Dense(1)(capa)
        out = tf.subtract(v, adv)
        
        return Model(inp, out)

    def experience(self, s, a, r, s1, done=False):
        if done:
            s1 = None
        self.buffer.append((s, a, r, s1))

    def sample(self):
        # Standard Replay
        # idx = np.random.choice(
        #     len(self.buffer),
        #     size=self.batch_size,
        #     replace=False
        # )

        # CER
        idx = np.hstack([np.random.choice(
                            (len(self.buffer)-1),
                            size=self.batch_size - 1,
                            replace=False),(len(self.buffer)-1)])

        return [self.buffer[i] for i in idx]

    def encode(self, s):
        if self.minus_factor is not None:
            s = s - self.minus_factor
        if self.div_factor is not None:
            s = np.divide(s, self.div_factor)
        return s
    
    @staticmethod
    def getQs(s, model):
        return model(s).numpy()

    def act(self, s):
        if np.random.rand() > self.epsilon:
            a = self.act_no_explore(s)
        else: 
            a = self.env.action_space.sample()
        return a

    def act_no_explore(self, s):
        q = self.getQs(s.reshape(1, -1), self.primary_model)
        return np.argmax(q)

    # update target with primary model weights
    def update_target(self):
        self.target_model.set_weights(self.primary_model.get_weights())
            
    def learn(self):
        batch = self.sample()
        s_list = np.array([experience[0].tolist() for experience in batch])
        s1_list = np.array([
            experience[3].tolist() for experience in batch 
            if experience[3] is not None
        ])
        qs = self.getQs(s_list, self.primary_model)
        # Primary model selects the best next action...
        qs1 = self.getQs(s1_list, self.primary_model)
        # ...and target model evaluates it
        q_target = self.getQs(s1_list, self.target_model)
        targets = qs

        k = 0
        for i, (s, a, r, s1) in enumerate(batch):
            if s1 is None:
                targets[i][a] = r
            else:
                # Standard DDQL
                # best_next_action = np.argmax(qs1[k])
                # targets[i][a] = r + self.gamma * q_target[k, best_next_action]
                # k += 1
                
                # Expected DDQL
                best_next_action = np.argmax(qs1[k])
                max_value = q_target[k, best_next_action] * (1 - self.epsilon)
                mean_value = np.mean(q_target[k, :]) * (self.epsilon)
                exp_value = max_value + mean_value
                targets[i][a] = r + self.gamma * exp_value
                k += 1   
                         
        loss = self.primary_model.train_on_batch(s_list, targets)

        return loss

In [ ]:
# Set parameters
nepisodes = 1000
n_log = 25

epsilon = 1
min_epsilon = 0.05
epsilon_decay = (epsilon - min_epsilon) / (nepisodes * 0.9)
gamma = 0.995
alpha = 0.0001
buffer_size = 10000
batch_size = 64
hidden_neurons = 120
learn_after = batch_size
update_target_after = 10 * learn_after

# Init env and agent
env = gym.make('LunarLander-v2')

agent = RainbowDQNAgent(
    env, gamma=gamma, epsilon=epsilon, alpha=alpha,
    batch_size=batch_size, buffer_size=buffer_size,
    hidden_neurons=hidden_neurons
)

# Collect Experiences
s = env.reset() 
for exps in range(buffer_size):
    a = agent.act(s)
    s1, r, done, _ = env.step(a)
    agent.experience(s, a, r, s1, done)
    s = s1
    if not exps % 5000:
        print(f'buffer exps: {exps}')
    if done:
        s = env.reset()

# Train Algo
learn_counter = 0
history_steps = []
history_rewards = []
history_disc_rewards = []
history_losses = []
for episode in range(nepisodes):
    s = env.reset()
    step = 0
    cum_reward = 0
    dis_cum_reward = 0
    episode_losses = []
    while True:
        a = agent.act(s)
        s1, r, done, _ = env.step(a)
        agent.experience(s, a, r, s1, done)
        learn_counter += 1
        cum_reward += r
        dis_cum_reward += agent.gamma ** step * r
        s = s1
        step += 1
        if not learn_counter % learn_after:
            mse = agent.learn()
        if not learn_counter % update_target_after:
            agent.update_target()
        if done:
            agent.epsilon = max([agent.epsilon - epsilon_decay, min_epsilon])
            history_rewards.append(cum_reward)
            history_disc_rewards.append(dis_cum_reward)
            history_losses.append(mse)
            history_steps.append(step)
            if not episode % n_log:
                mse = agent.learn()
                print(
                    f'Episode: {episode}, '
                    f'steps: {np.round(np.mean(history_steps[-n_log:]), 2)}, '
                    f'rew: {np.round(np.mean(history_rewards[-n_log:]), 2)}, '
                    f'mse: {np.round(mse)}, '
                    f'eps: {np.round(agent.epsilon, 2)}'
                )
            break    

In [ ]:
plot_training_results(
    rewards=history_rewards,
    losses=history_losses,
    window=10,
    scale="linear"
)

In [ ]:
env = gym.make('LunarLander-v2')
s = env.reset()
img = plt.imshow(env.render(mode='rgb_array'))
cum_reward = 0
done = False
step = 0
while True:
    img.set_data(env.render(mode='rgb_array'))
    plt.title(f'Step: {step}, rew: {np.round(cum_reward)}')
    plt.axis('off')
    display.display(plt.gcf())
    display.clear_output(wait=True)
    if done:
        break
    action = agent.act_no_explore(s)
    s, r, done, _ = env.step(action)
    step += 1
    cum_reward += agent.gamma ** step * r

env.close()
print(f"Done Rew: {np.round(agent.gamma ** step * r, 2)}")